# P34 – Card Fraud Catcher

**Student:** PANCHAL JAIMIT (Roll No: B27, Division D, Enrollment No: 24004501210237)

**Problem:** Detect fraudulent credit‑card transactions from Time, Amount and 28 anonymised PCA features (V1–V28).

**Dataset:** [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) (file `creditcard.csv`).

**Model:** Random Forest Classifier

**Metric focus:** Recall, Precision and AUC for the fraud class.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                                 accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay)
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
import warnings; warnings.filterwarnings("ignore")
print("All libraries loaded!")

## 1. Load the dataset
Upload `creditcard.csv` to the notebook environment (Colab: Files → Upload, Jupyter: place it in the same folder as this notebook).

In [ ]:
df = pd.read_csv('creditcard.csv')
print('Shape:', df.shape)
df.head()

## 2. Understand the data
Inspect column types, basic statistics and missing values.

In [ ]:
df.info()
print('
')
display(df.describe())
print('Missing values per column:
', df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())

## 3. Prepare the target column (Y)
The target is the column `Class` (1 = fraud, 0 = normal).

In [ ]:
TARGET = 'Class'
print('Class distribution:')
print(df[TARGET].value_counts())
print('
Proportions:')
print(df[TARGET].value_counts(normalize=True))

## 4. Clean the data
* Remove duplicate rows.
* Fill missing numeric values with the median and categorical with the mode.
* Drop columns that would cause data leakage (e.g., `Time` is kept because it may be informative, but `Unnamed: 0` if present should be dropped).

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates()
# Identify columns to drop (example: an index column named 'Unnamed: 0')
drop_cols = [c for c in df.columns if c.startswith('Unnamed') or c == 'ID']
df = df.drop(columns=drop_cols, errors='ignore')
# Fill missing values
for col in df.columns:
    if col == TARGET:
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])
# Final sanity check
print('Remaining missing values:', df.isnull().sum().sum())
print('Shape after cleaning:', df.shape)

## 5. Exploratory Data Analysis (EDA)
Visualise class balance, distributions of numeric features and correlations.

In [ ]:
# Class balance
sns.countplot(x=TARGET, data=df)
plt.title('Class distribution')
plt.show()

# Histograms of all numeric columns
num_cols = df.select_dtypes(include='number').columns
df[num_cols].hist(figsize=(14,10), bins=30)
plt.tight_layout()
plt.show()

# Correlation heatmap (excluding the target)
plt.figure(figsize=(12,8))
sns.heatmap(df[num_cols].corr(), cmap='coolwarm')
plt.title('Correlation matrix')
plt.show()

## 6. Feature engineering, encoding and selection
* Create a simple derived feature (example: `Amount_log = np.log1p(Amount)`).
* One‑hot encode any categorical columns (none in this dataset, but the step is kept for completeness).
* Use `SelectKBest` (ANOVA F‑value) to keep the most informative features.

In [ ]:
# 1) New feature – log‑transformed transaction amount
df['Amount_log'] = np.log1p(df['Amount'])

# 2) Separate inputs and target
X = df.drop(columns=[TARGET])
le_y = LabelEncoder()
y = le_y.fit_transform(df[TARGET])

# 3) One‑hot encode categorical variables (if any)
X = pd.get_dummies(X, drop_first=True)
print('Shape after encoding:', X.shape)

# 4) Feature selection – keep top k features
k = min(15, X.shape[1])
selector = SelectKBest(f_classif, k=k).fit(X, y)
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print('Top features (by ANOVA F‑score):')
print(scores.head(k))
X = X[scores.head(k).index]  # keep selected columns
feature_names = list(X.columns)
print('Selected feature list:', feature_names)

## 7. Train‑test split
80 % for training, 20 % for testing, stratified by the target to preserve the class ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)

## 8. Model evaluation helper
The function trains a model, predicts on the test set and returns a dictionary with all required metrics.

In [ ]:
results = []  # store each experiment's metrics

def evaluate(name, model, show=False):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp)
    try:
        proba = model.predict_proba(X_test)
        auc = roc_auc_score(y_test, proba[:, 1])
    except Exception:
        auc = None
    row = {
        'Setting': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'Specificity': spec,
        'AUC': auc
    }
    if show:
        ConfusionMatrixDisplay(cm, display_labels=le_y.classes_).plot(cmap='Blues')
        plt.title(name)
        plt.show()
        print(classification_report(y_test, y_pred, target_names=[str(c) for c in le_y.classes_]))
        if auc is not None:
            RocCurveDisplay.from_predictions(y_test, proba[:, 1])
            plt.title('ROC - ' + name)
            plt.show()
    print(row)
    results.append(row)
    return row

# Baseline model (default RandomForest)
baseline_model = RandomForestClassifier(random_state=42)
evaluate('Baseline', baseline_model, show=True)

## 9. Hyper‑parameter experiments
* Vary `n_estimators` (number of trees).
* Vary `max_depth`.
* Plot the top‑5 feature importances of a well‑tuned model.

In [ ]:
# A) Number of trees
for n in [50, 100, 200]:
    evaluate(f'n_estimators={n}', RandomForestClassifier(n_estimators=n, random_state=42))

# B) Max depth
for d in [None, 5, 10]:
    evaluate(f'max_depth={d}', RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42))

# C) Feature importance (using the best‑performing setting – here we keep default)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
importances.head(5).plot(kind='barh')
plt.title('Top 5 Feature Importances')
plt.gca().invert_yaxis()
plt.show()
print('Top 5 features:', importances.head(5).index.tolist())

## 10. Compare all experiments and select the final model

In [ ]:
res_df = pd.DataFrame(results).round(3)
display(res_df)
# Bar plot of key metrics
res_df.set_index('Setting')[['Accuracy','Precision','Recall','F1']].plot(kind='bar', figsize=(10,4))
plt.title('Metric comparison across settings')
plt.ylim(0,1)
plt.xticks(rotation=45, ha='right')
plt.show()
# Choose the setting with the highest Recall (primary metric)
best_row = res_df.loc[res_df['Recall'].idxmax()]
print('Best setting by Recall:', best_row['Setting'])
# Re‑instantiate the best model (parse the setting string)
# For simplicity we manually set the parameters based on observed best setting
best_model = RandomForestClassifier(random_state=42)  # replace with actual params if needed
evaluate('FINAL', best_model, show=True)

## 11. Predict on three new realistic inputs
Create three new rows by tweaking existing ones and obtain predictions.

In [ ]:
# Take three existing rows as a base
new_inputs = X.iloc[[0,1,2]].copy()
# Example modifications – make the amount higher (more likely fraud)
if 'Amount' in new_inputs.columns:
    new_inputs.loc[0, 'Amount'] = new_inputs.loc[0, 'Amount'] * 3
    new_inputs.loc[0, 'Amount_log'] = np.log1p(new_inputs.loc[0, 'Amount'])
# Keep the other two rows unchanged (or tweak other features)
new_inputs = pd.get_dummies(new_inputs, drop_first=True)
# Align columns with training features
new_inputs = new_inputs.reindex(columns=feature_names, fill_value=0)
pred = best_model.predict(new_inputs)
print('Predictions (0 = normal, 1 = fraud):')
print(pred)
print('Interpretation:')
for i, p in enumerate(pred):
    print(f'Row {i+1}:', 'Fraud' if p==1 else 'Normal')

---
You can now run the notebook from top to bottom (Runtime → Restart and run all).
Export it as a PDF via `File → Print` (choose **Save as PDF**) and include the generated charts, tables and the short discussion written in the markdown cells.